# PCA Process Monitor — Processo Continuo

Dimostrazione completa del package `pca-process-monitor` su un dataset sintetico
di un impianto chimico continuo.

**Flusso:**
1. Setup e caricamento dati
2. Overview e statistica descrittiva
3. Preprocessing e split calibrazione/test
4. Costruzione modello NIPALS — selezione numero PC
5. Score plot, loading plot, biplot, heatmap
6. Diagnostica: T², Q, contribution plots
7. AI interpreter (opzionale)


## 0 — Setup

In [ ]:
# Clona la repo e installa dipendenze
import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO = 'pca-process-monitor'
USER = 'MarDan93'

if not os.path.exists(f'/content/{REPO}'):
    os.system(f'git clone https://{GITHUB_TOKEN}@github.com/{USER}/{REPO}.git')

os.chdir(f'/content/{REPO}')
print(f'Working dir: {os.getcwd()}')

In [ ]:
!pip install -q numpy pandas matplotlib scipy plotly

In [ ]:
import sys
sys.path.insert(0, '/content/pca-process-monitor')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

import pca_monitor as pca
from pca_monitor.nipals         import NIPALS
from pca_monitor.preprocessing  import (
    detect_process_type, missing_summary, handle_missing,
    split_calibration_test, dataframe_info, get_numeric_columns
)
from pca_monitor.diagnostics    import (
    ContinuousMonitor, find_anomalies, compute_cal_contribution_limits
)
from pca_monitor import plots

print('✓ Package caricato correttamente')

## 1 — Generazione e caricamento dati

In [ ]:
# Genera i dataset sintetici
!python data/generate_datasets.py

In [ ]:
# Carica calibrazione e test
df_cal  = pd.read_csv('data/continuous_calibration.csv', index_col='obs_id')
df_test = pd.read_csv('data/continuous_test.csv',        index_col='obs_id')

print('=== CALIBRAZIONE ===')
print(f'Shape: {df_cal.shape}')
print(df_cal.head())
print('\n=== TEST ===')
print(f'Shape: {df_test.shape}')
print(df_test.head())

## 2 — Overview e statistica descrittiva

In [ ]:
# Rilevamento automatico tipo processo
detection = detect_process_type(df_cal)
print(f"Tipo rilevato  : {detection['type']}")
print(f"Confidenza     : {detection['confidence']}")
print(f"Motivazione    : {detection['reason']}")

# Override manuale se necessario
PROCESS_TYPE = detection['type']  # cambia in 'batch' o 'continuous' se sbagliato
print(f"\nTipo usato     : {PROCESS_TYPE}")

In [ ]:
# Info generale dataset
info = dataframe_info(df_cal)
for k, v in info.items():
    print(f'{k:20s}: {v}')

In [ ]:
# Statistica descrittiva
VAR_COLS = get_numeric_columns(df_cal, exclude=['split'])
print(f'Variabili di processo: {VAR_COLS}')
df_cal[VAR_COLS].describe().round(3)

In [ ]:
# Grafici distribuzione — cambia kind in 'boxplot' o 'timeseries'
fig = plots.plot_variable_distributions(df_cal, VAR_COLS, kind='histogram')
plt.show()

In [ ]:
# Statistica descrittiva visuale
fig = plots.plot_descriptive_stats(df_cal, VAR_COLS)
plt.show()

In [ ]:
# Missing values
ms = missing_summary(df_cal)
print(ms)
fig = plots.plot_missing_heatmap(df_cal)
plt.show()

## 3 — Preprocessing

In [ ]:
# Gestione missing (se presenti)
df_cal_clean  = handle_missing(df_cal,  method='mean', columns=VAR_COLS)
df_test_clean = handle_missing(df_test, method='mean', columns=VAR_COLS)

# Matrici numpy per il modello
X_cal  = df_cal_clean[VAR_COLS].values
X_test = df_test_clean[VAR_COLS].values

print(f'X_cal  shape: {X_cal.shape}')
print(f'X_test shape: {X_test.shape}')

## 4 — Costruzione modello NIPALS e selezione PC

In [ ]:
# Fit modello con numero massimo di PC
MAX_PC = min(6, X_cal.shape[1])
model  = NIPALS(n_components=MAX_PC, scale='auto', tol=1e-6, max_iter=500)
model.fit(X_cal)
model.summary()

In [ ]:
# Scree plot + varianza cumulativa
fig = plots.plot_scree(
    eigenvalues   = model.eigenvalues,
    explained_var = model.explained_variance_ratio_,
)
plt.show()

In [ ]:
# RMSECV — metodo più accurato per selezione PC
# Nota: può richiedere qualche secondo su dataset grandi
print('Calcolo RMSECV in corso...')
rmsecv_vals, optimal_nc = model.rmsecv(X_cal, max_components=MAX_PC, cv_folds=5)

print(f'\nRMSECV per PC: {[round(v,4) for v in rmsecv_vals]}')
print(f'Numero ottimale di PC (RMSECV): {optimal_nc}')

fig = plots.plot_rmsecv(rmsecv_vals, optimal_nc)
plt.show()

In [ ]:
# Scree plot aggiornato con PC selezionate
fig = plots.plot_scree(
    eigenvalues   = model.eigenvalues,
    explained_var = model.explained_variance_ratio_,
    n_selected    = optimal_nc
)
plt.show()

# Puoi sovrascrivere il numero di PC qui se preferisci
N_PC = optimal_nc
print(f'PC usate per la diagnostica: {N_PC}')

## 5 — Analisi scores, loadings, biplot

In [ ]:
# Score plot calibrazione — seleziona la coppia di PC
PC_X, PC_Y = 1, 2  # modifica qui per vedere altre coppie

fig = plots.plot_scores(
    T      = model.T,
    pc_x   = PC_X,
    pc_y   = PC_Y,
    alpha  = 0.05,
    title  = 'Score plot — calibrazione'
)
plt.show()

In [ ]:
# Loading plot
fig = plots.plot_loadings(
    P         = model.P,
    var_names = VAR_COLS,
    pc_x      = PC_X,
    pc_y      = PC_Y
)
plt.show()

In [ ]:
# Biplot
fig = plots.plot_biplot(
    T         = model.T,
    P         = model.P,
    var_names = VAR_COLS,
    pc_x      = PC_X,
    pc_y      = PC_Y
)
plt.show()

In [ ]:
# Heatmap loadings
fig = plots.plot_loading_heatmap(
    P           = model.P,
    var_names   = VAR_COLS,
    n_components= N_PC
)
plt.show()

In [ ]:
# Analisi residui per variabile selezionata
_, E_cal = model.compute_Q(X_cal, N_PC)
VAR_IDX  = 0  # indice variabile da analizzare — modifica qui

fig = plots.plot_residuals(E_cal, VAR_COLS, var_idx=VAR_IDX)
plt.show()

## 6 — Diagnostica sul set di test

In [ ]:
# Inizializza il monitor
monitor = ContinuousMonitor(
    model        = model,
    n_components = N_PC,
    alpha        = 0.05,
    var_names    = VAR_COLS
)

print(f'Limite T² (95%): {monitor.T2_lim:.4f}')
print(f'Limite Q  (95%): {monitor.Q_lim:.4f}')

In [ ]:
# Monitoraggio set di test
result = monitor.monitor(X_test)
print(monitor.summary_report(result))

In [ ]:
# Grafico T² e Q
anomaly_idx = result['anomaly_any'].nonzero()[0].tolist()

fig = plots.plot_T2_Q(
    T2            = result['T2'],
    Q             = result['Q'],
    T2_limit      = result['T2_limit'],
    Q_limit       = result['Q_limit'],
    highlight_idx = anomaly_idx,
    title         = 'Monitoraggio set di test'
)
plt.show()

In [ ]:
# Score plot test con anomalie evidenziate
T_test = model.transform(X_test, N_PC)

fig = plots.plot_scores(
    T             = T_test,
    pc_x          = PC_X,
    pc_y          = PC_Y,
    highlight_idx = anomaly_idx,
    alpha         = 0.05,
    title         = 'Score plot — set di test'
)
plt.show()

In [ ]:
# Contribution plot per la prima osservazione anomala
if anomaly_idx:
    obs = anomaly_idx[0]
    ca  = monitor.contribution_analysis(X_test, obs)

    print(f'Osservazione anomala: {obs}')
    print(f'Top variabili T²: {ca["top_vars_T2"]}')
    print(f'Top variabili Q : {ca["top_vars_Q"]}')

    # Contribution T²
    ct2_all = model.contributions_T2(X_test, N_PC)
    fig = plots.plot_contributions(
        contrib   = ct2_all,
        var_names = VAR_COLS,
        obs_idx   = obs,
        stat      = 'T2',
        ref_limit = ca['ref_T2']
    )
    plt.show()

    # Contribution Q
    cq_all, _ = model.contributions_Q(X_test, N_PC)
    fig = plots.plot_contributions(
        contrib   = cq_all,
        var_names = VAR_COLS,
        obs_idx   = obs,
        stat      = 'Q',
        ref_limit = ca['ref_Q']
    )
    plt.show()
else:
    print('Nessuna anomalia rilevata nel set di test.')

## 7 — AI Interpreter (opzionale)

In [ ]:
# Inizializza il contesto con i dati reali del modello
from pca_monitor.ai_interpreter import PCAContext, create_interpreter

ctx = PCAContext()
ctx.update(
    process_type   = PROCESS_TYPE,
    section        = 'diagnostics',
    n_samples      = X_cal.shape[0],
    n_vars         = X_cal.shape[1],
    var_names      = VAR_COLS,
    n_components   = N_PC,
    scale_method   = 'auto',
    explained_var  = (model.explained_variance_ratio_[:N_PC] * 100).tolist(),
    cumulative_var = float(model.explained_variance_ratio_[:N_PC].sum() * 100),
    rmsecv_values  = rmsecv_vals.tolist(),
    optimal_nc_rmsecv = optimal_nc,
    T2_limit       = monitor.T2_lim,
    Q_limit        = monitor.Q_lim,
    anomaly_summary = find_anomalies(result['T2'], result['Q'],
                                     monitor.T2_lim, monitor.Q_lim),
)

# Crea interprete (auto: prova Gemini, poi Claude)
ai = create_interpreter(context=ctx, backend='auto')
print('Contesto AI pronto.')

In [ ]:
# Esempio domanda — modifica liberamente
if ai:
    ai.ask('Quante PC sono state selezionate e perché? '
           'Il numero scelto da RMSECV è coerente con lo scree plot?')

In [ ]:
# Seconda domanda — la memoria della conversazione è mantenuta
if ai:
    ai.ask('Quali variabili sembrano più problematiche '
           'nelle osservazioni anomale rilevate?')